# Intrusion Detection on CICIDS2017 — Reproduction & Critique

**Course:** Data Science Methods in Cyber Security (Dr. Uri Itai, University of Haifa)  
**Student:** Eyal Steinmetz  
**Deadline:** Friday, 10 July 2026, 23:59

**Mission:** Reproduce Rodríguez et al. (2022, *Sensors* MDPI), who claim a Random Forest reaches
**F1 > 0.999** on CICIDS2017, then demonstrate that the claim is an artifact of three documented
methodological flaws — duplicate contamination, temporal leakage, and aggregate-only metrics —
using Engelen et al. (2021) as peer-reviewed counter-evidence.

| Notebook section | Report section | Purpose |
|---|---|---|
| 0. Setup | front matter | Imports, seed, paths, figure helper |
| **1. Data Loading & Cleaning** | §3 Data | Load 8 day-files, tag `day`, strip columns, quantify quality issues |
| **2. EDA** | §4 EDA | Class balance, temporal structure, Spearman correlation |
| **3. Feature Engineering** | §5 | Encoding, log-transform, correlation pruning (78 → 40 features) |
| 4. Model Training | §6 | Strategy A (random) vs Strategy B (dedup + temporal) |
| 5. Evaluation | §6/§7 | Metric suite, confusion matrices, ROC/PR |
| 6. Error Analysis & Critique | §2 | Three-flaw framework, per-class failure, verdict |
| 7. Executive Summary | §1/§8 | Problem → method → findings → verdict |

> Sections **0–3** are implemented (Setup, Data Loading & Cleaning, EDA, Feature Engineering) — Phases 2–4 in the project plan. Sections 4–7 (modelling, evaluation, critique) follow.

## Section 0 — Setup & Imports

We fix a single global random seed (`RANDOM_STATE = 42`) used by every split, model, and SMOTE
call for reproducibility, define the project paths, and provide a `save_fig()` helper so every
figure lands in `figures/` at a consistent resolution. The notebook lives in `notebooks/`, so
all paths are relative to that directory.

In [1]:
# Standard library
import warnings
from pathlib import Path

# Third-party (data + visualization)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Reproducibility — one seed used everywhere
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths (notebook runs from notebooks/)
DATA_DIR = Path("../data/MachineLearningCVE")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Warnings policy: silence only FutureWarning noise from pandas/sklearn version churn.
# We do NOT silence SettingWithCopyWarning — that one signals real bugs and must surface.
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid")


def save_fig(fig, name, tight=True):
    """Save a figure to figures/ with consistent settings, then close it."""
    path = FIGURES_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight" if tight else None)
    plt.close(fig)
    print(f"Saved: {path}")


print("Imports OK")
print(f"RANDOM_STATE = {RANDOM_STATE}")
print(f"DATA_DIR     = {DATA_DIR.resolve()}")
print(f"FIGURES_DIR  = {FIGURES_DIR.resolve()}")

Imports OK
RANDOM_STATE = 42
DATA_DIR     = C:\Users\אייל. ש\Documents\HUP\cyber\project\data\MachineLearningCVE
FIGURES_DIR  = C:\Users\אייל. ש\Documents\HUP\cyber\project\figures


## Section 1 — Data Loading & Cleaning

We load CICIDS2017 from its pre-extracted CSV flow features (`MachineLearningCVE/`). The published
release ships **8** CSV files, not 5: Thursday is split into *WebAttacks* and *Infiltration*
captures, and Friday into *Morning*, *PortScan*, and *DDoS* captures. Because the ML CSVs carry no
timestamp column, we derive the weekday — needed later for the temporal split (Flaw 2) — directly
from each filename.

### 1.1 — Load all day-files and concatenate

Each CSV is one capture. We tag every row with its source weekday (`day_name`) and an ordinal
`day` index (Mon=0 … Fri=4) *before* concatenating, then strip whitespace from column names
immediately — CICFlowMeter emits leading/trailing spaces in headers (Engelen et al., 2021).

In [2]:
# Ordinal weekday encoding for the temporal split (Mon=0 .. Fri=4)
WEEKDAY_ORDER = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3, "Friday": 4}


def weekday_from_filename(filename: str):
    """Derive (weekday_name, ordinal) from a CICIDS2017 CSV filename.

    The 8-file release prefixes every file with the weekday, e.g.
    'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv'. Case-insensitive
    because the Wednesday file uses a lowercase 'workingHours'.
    """
    low = filename.lower()
    for name, idx in WEEKDAY_ORDER.items():
        if low.startswith(name.lower()):
            return name, idx
    raise ValueError(f"Cannot derive weekday from filename: {filename}")


csv_paths = sorted(DATA_DIR.glob("*.pcap_ISCX.csv"))
assert csv_paths, f"No CSVs found in {DATA_DIR.resolve()} — place the MachineLearningCVE files there."

frames = []
for path in csv_paths:
    day_name, day_idx = weekday_from_filename(path.name)
    part = pd.read_csv(path, low_memory=False)
    part["day"] = day_idx
    part["day_name"] = day_name
    frames.append(part)
    print(f"  {path.name:<55} -> {day_name:<9} ({len(part):>8,} rows)")

df = pd.concat(frames, ignore_index=True)
df.columns = df.columns.str.strip()  # strip once: CICFlowMeter headers carry whitespace

print(f"\nLoaded {len(df):,} rows x {len(df.columns)} columns from {len(csv_paths)} files")
print("\nRows per day:")
print(df["day_name"].value_counts().reindex(WEEKDAY_ORDER.keys()).to_string())

  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv        -> Friday    ( 225,745 rows)


  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    -> Friday    ( 286,467 rows)


  Friday-WorkingHours-Morning.pcap_ISCX.csv               -> Friday    ( 191,033 rows)


  Monday-WorkingHours.pcap_ISCX.csv                       -> Monday    ( 529,918 rows)


  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv -> Thursday  ( 288,602 rows)


  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv  -> Thursday  ( 170,366 rows)


  Tuesday-WorkingHours.pcap_ISCX.csv                      -> Tuesday   ( 445,909 rows)


  Wednesday-workingHours.pcap_ISCX.csv                    -> Wednesday ( 692,703 rows)



Loaded 2,830,743 rows x 81 columns from 8 files

Rows per day:
day_name
Monday       529918
Tuesday      445909
Wednesday    692703
Thursday     458968
Friday       703245


### 1.2 — First inspection: dtypes, columns, labels

We confirm the column names are clean, locate the only expected non-numeric feature columns
(`Label` plus our two tag columns), flag the known redundant `Fwd Header Length.1` duplicate
column, and print the full label distribution. The label counts must show the severe imbalance
(~80% BENIGN) and the ultra-rare attack classes (Heartbleed, SQL Injection, Infiltration) that
later expose Flaw 3.

In [3]:
# Non-numeric columns (expect only Label + our two tag columns)
non_numeric = df.select_dtypes(exclude=np.number).columns.tolist()
print(f"Non-numeric columns ({len(non_numeric)}): {non_numeric}")

# Known redundant column from CICFlowMeter (Engelen et al., 2021) — flag now, drop in Phase 3
dup_col = "Fwd Header Length.1"
print(f"\n'{dup_col}' present (redundant, drop later): {dup_col in df.columns}")

print(f"\nTotal feature columns (excl. Label/day/day_name): {len(df.columns) - 3}")
print("\nAll column names:")
for i, c in enumerate(df.columns):
    print(f"  [{i:>2}] {c}")

Non-numeric columns (2): ['Label', 'day_name']

'Fwd Header Length.1' present (redundant, drop later): True

Total feature columns (excl. Label/day/day_name): 78

All column names:
  [ 0] Destination Port
  [ 1] Flow Duration
  [ 2] Total Fwd Packets
  [ 3] Total Backward Packets
  [ 4] Total Length of Fwd Packets
  [ 5] Total Length of Bwd Packets
  [ 6] Fwd Packet Length Max
  [ 7] Fwd Packet Length Min
  [ 8] Fwd Packet Length Mean
  [ 9] Fwd Packet Length Std
  [10] Bwd Packet Length Max
  [11] Bwd Packet Length Min
  [12] Bwd Packet Length Mean
  [13] Bwd Packet Length Std
  [14] Flow Bytes/s
  [15] Flow Packets/s
  [16] Flow IAT Mean
  [17] Flow IAT Std
  [18] Flow IAT Max
  [19] Flow IAT Min
  [20] Fwd IAT Total
  [21] Fwd IAT Mean
  [22] Fwd IAT Std
  [23] Fwd IAT Max
  [24] Fwd IAT Min
  [25] Bwd IAT Total
  [26] Bwd IAT Mean
  [27] Bwd IAT Std
  [28] Bwd IAT Max
  [29] Bwd IAT Min
  [30] Fwd PSH Flags
  [31] Bwd PSH Flags
  [32] Fwd URG Flags
  [33] Bwd URG Flags
  [34] Fwd H

In [4]:
# Normalize label text: the raw CSVs encode the web-attack dash as a non-UTF-8 byte that
# reads back as the replacement char (e.g. "Web Attack � XSS"). Map it to a plain hyphen
# and strip whitespace so per-class reporting (Flaw 3) uses clean, stable class names.
df["Label"] = df["Label"].str.replace("�", "-", regex=False).str.replace("  ", " ").str.strip()

# Label distribution — counts and percentages
label_counts = df["Label"].value_counts()
label_pct = df["Label"].value_counts(normalize=True) * 100
label_summary = pd.DataFrame({"count": label_counts, "pct": label_pct.round(4)})
print(label_summary.to_string())

benign_pct = label_pct.get("BENIGN", 0.0)
print(f"\nBENIGN share: {benign_pct:.1f}%   |   Attack share: {100 - benign_pct:.1f}%")
print(f"Distinct labels: {df['Label'].nunique()}")

                              count      pct
Label                                       
BENIGN                      2273097  80.3004
DoS Hulk                     231073   8.1630
PortScan                     158930   5.6144
DDoS                         128027   4.5227
DoS GoldenEye                 10293   0.3636
FTP-Patator                    7938   0.2804
SSH-Patator                    5897   0.2083
DoS slowloris                  5796   0.2048
DoS Slowhttptest               5499   0.1943
Bot                            1966   0.0695
Web Attack - Brute Force       1507   0.0532
Web Attack - XSS                652   0.0230
Infiltration                     36   0.0013
Web Attack - Sql Injection       21   0.0007
Heartbleed                       11   0.0004

BENIGN share: 80.3%   |   Attack share: 19.7%


Distinct labels: 15


### 1.3 — Quality issue: Inf and NaN values

CICFlowMeter divides by flow duration to produce `Flow Bytes/s` and `Flow Packets/s`; zero-duration
flows yield `Inf` (Engelen et al., 2021). We count these, convert `±Inf` to `NaN`, report which
columns carry missing values, and drop the affected rows. This minimal cleaning is identical for
both evaluation strategies, so doing it once here introduces no leakage.

In [5]:
numeric_cols = df.select_dtypes(include=np.number).columns

# 1. Locate Inf cells once, then reuse the same mask to set them to NaN.
#    (One numeric-only scan — no second full-frame df.replace pass.)
inf_mask = np.isinf(df[numeric_cols])
inf_total = int(inf_mask.to_numpy().sum())
inf_by_col = inf_mask.sum()
inf_by_col = inf_by_col[inf_by_col > 0]
print(f"Total Inf/-Inf cells: {inf_total:,}")
print("Columns carrying Inf:")
print(inf_by_col.to_string())

df[numeric_cols] = df[numeric_cols].mask(inf_mask)  # True -> NaN, reusing the mask

# 2. NaN counts per column (after Inf conversion)
nan_counts = df.isna().sum()
nan_cols = nan_counts[nan_counts > 0]
print(f"\nColumns with NaN (post Inf->NaN): {len(nan_cols)}")
print(nan_cols.to_string())

# 3. Drop rows with any NaN (document the count)
n_before = len(df)
df = df.dropna().reset_index(drop=True)
print(f"\nDropped {n_before - len(df):,} rows containing NaN ({(n_before - len(df)) / n_before * 100:.3f}%)")
print(f"Remaining rows: {len(df):,}")

Total Inf/-Inf cells: 4,376
Columns carrying Inf:
Flow Bytes/s      1509
Flow Packets/s    2867



Columns with NaN (post Inf->NaN): 2
Flow Bytes/s      2867
Flow Packets/s    2867



Dropped 2,867 rows containing NaN (0.101%)
Remaining rows: 2,827,876


In [6]:
# Negative values in flow features that are physically non-negative (counts, lengths, durations).
# CICFlowMeter is known to emit spurious negatives (Engelen et al., 2021); we document, not fix, here.
feature_cols = [c for c in numeric_cols if c not in ("day",)]
neg_counts = (df[feature_cols] < 0).sum()
neg_cols = neg_counts[neg_counts > 0].sort_values(ascending=False)
print(f"Feature columns containing negative values: {len(neg_cols)}")
print(neg_cols.to_string() if len(neg_cols) else "  (none)")

Feature columns containing negative values: 13
Init_Win_bytes_backward    1439672
Init_Win_bytes_forward     1001172
Flow IAT Min                  2890
Flow Packets/s                 115
Flow Duration                  115
Flow IAT Mean                  115
Flow IAT Max                   115
Flow Bytes/s                    85
Fwd Header Length               35
Fwd Header Length.1             35
min_seg_size_forward            35
Bwd Header Length               22
Fwd IAT Min                     17


### 1.4 — Quality issue: duplicate rows (Flaw 1)

CICIDS2017 contains a large number of **exact duplicate flow records** (Engelen et al., 2021). Under
a random train/test split these duplicates are scattered across both sets, letting the model
"memorize" test points it has already seen in training — inflating every metric. This is **Flaw 1**
of our critique.

We **quantify** the duplicates here and persist the count to `figures/dedup_stats.txt`, but we do
**not** remove them yet. Deduplication is deferred to **Strategy B** so we can measure the exact
F1 inflation that the duplicates cause (run A0 with duplicates vs A1 without).

In [7]:
# Duplicates over the feature + label columns (ignore the tag columns we added).
dedup_subset = [c for c in df.columns if c not in ("day", "day_name")]
n_dupes = int(df.duplicated(subset=dedup_subset).sum())
pct_dupes = n_dupes / len(df) * 100
print(f"FLAW 1 - exact duplicate rows: {n_dupes:,} ({pct_dupes:.2f}% of data)")
print("Duplicates are RETAINED here on purpose; removal is deferred to Strategy B (A0 vs A1).")

# Persist for the report (cited verbatim in the critique)
stats_path = FIGURES_DIR / "dedup_stats.txt"
stats_path.write_text(
    f"rows_total={len(df)}\n"
    f"duplicate_rows={n_dupes}\n"
    f"duplicate_pct={pct_dupes:.4f}\n",
    encoding="utf-8",
)
print(f"Saved: {stats_path}")

FLAW 1 - exact duplicate rows: 307,078 (10.86% of data)
Duplicates are RETAINED here on purpose; removal is deferred to Strategy B (A0 vs A1).
Saved: ..\figures\dedup_stats.txt


### 1.5 — Zero-variance columns, artifacts, memory, and dtype downcast

We list **zero-variance** feature columns (constant for every row) — they carry no signal and are
dropped in Phase 3 — count the `Flow Duration == 0` CICFlowMeter artifacts, and report the
DataFrame's memory footprint. We then **downcast** the numeric columns to shrink the ~2 GB
footprint (ADR-10 / TODO P2).

**Ordering matters:** the downcast runs *after* every quality count above (duplicates, negatives,
zero-variance). Casting `float64 → float32` could collapse two rows that differ only beyond
float32 precision into an exact duplicate, which would inflate the Flaw 1 number — so all reported
quality statistics are computed at full precision first. We let pandas pick the smallest *safe*
type per column (`pd.to_numeric(downcast=...)`); this keeps `Destination Port` (max 65535) in
`int32` rather than overflowing `int16`, and we assert min/max are preserved.

In [8]:
# Zero-variance feature columns (constant) — candidates to drop in Phase 3
feature_only = df.drop(columns=["Label", "day", "day_name"]).select_dtypes(include=np.number)
zero_var_cols = feature_only.columns[feature_only.nunique() <= 1].tolist()
print(f"Zero-variance (constant) feature columns: {len(zero_var_cols)}")
for c in zero_var_cols:
    print(f"  - {c}")

# Flow Duration == 0 artifacts
if "Flow Duration" in df.columns:
    n_zero_dur = int((df["Flow Duration"] == 0).sum())
    print(f"\nFlows with Flow Duration == 0 (CICFlowMeter artifact): {n_zero_dur:,}")

# Memory footprint BEFORE downcast (full float64/int64)
mem_before = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nDataFrame memory footprint (pre-downcast): {mem_before:,.1f} MB")

Zero-variance (constant) feature columns: 8
  - Bwd PSH Flags
  - Bwd URG Flags
  - Fwd Avg Bytes/Bulk
  - Fwd Avg Packets/Bulk
  - Fwd Avg Bulk Rate
  - Bwd Avg Bytes/Bulk
  - Bwd Avg Packets/Bulk
  - Bwd Avg Bulk Rate

Flows with Flow Duration == 0 (CICFlowMeter artifact): 0

DataFrame memory footprint (pre-downcast): 1,784.6 MB


In [9]:
# Safe numeric downcast (run AFTER all quality counts so float precision can't alter them).
# pd.to_numeric(downcast=...) chooses the smallest type that holds each column's range,
# so it never overflows (e.g. Destination Port max 65535 -> int32, not int16).
num_cols = df.select_dtypes(include=np.number).columns
ranges_before = df[num_cols].agg(["min", "max"])

for col in df.select_dtypes(include="integer").columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")
for col in df.select_dtypes(include="float").columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

# Verify the downcast preserved every column's value range (no overflow / no corruption).
ranges_after = df[num_cols].agg(["min", "max"])
assert np.allclose(ranges_before.to_numpy(), ranges_after.to_numpy(), rtol=1e-3, equal_nan=True), (
    "Downcast altered a column's min/max — aborting to avoid silent data corruption."
)

mem_after = df.memory_usage(deep=True).sum() / 1024**2
print(f"Memory footprint (post-downcast): {mem_after:,.1f} MB")
print(f"Reduction: {mem_before - mem_after:,.1f} MB ({(1 - mem_after / mem_before) * 100:.0f}% smaller)")
print(f"dtypes now: {df.select_dtypes(include=np.number).dtypes.value_counts().to_dict()}")

Memory footprint (post-downcast): 924.3 MB
Reduction: 860.3 MB (48% smaller)
dtypes now: {dtype('int32'): 27, dtype('int8'): 19, dtype('float64'): 15, dtype('float32'): 9, dtype('int16'): 7, dtype('int64'): 2}


### 1.6 — Data-quality summary

**Findings (all consistent with Engelen et al., 2021):**

- Column headers carried whitespace → stripped on load.
- `Flow Bytes/s` and `Flow Packets/s` carried `Inf` from divide-by-zero → converted to `NaN`; the
  affected rows were dropped (identical minimal cleaning for both strategies, so no leakage).
- A substantial fraction of rows are **exact duplicates** — the core leakage vector (Flaw 1). These
  are deliberately **kept** for now; the count is saved to `figures/dedup_stats.txt`.
- The label distribution is severely imbalanced (~80% BENIGN) with ultra-rare attack classes
  (Heartbleed, SQL Injection, Infiltration), foreshadowing Flaw 3.
- Zero-variance columns and the redundant `Fwd Header Length.1` are flagged for removal in Phase 3.

**Deferral note (deliberate):** Deduplication is *not* applied here. It is applied only inside
**Strategy B** so the difference between the duplicate-contaminated random split (A0) and the
deduplicated random split (A1) directly measures the F1 inflation caused by Flaw 1. This mirrors
the source paper's pipeline, which never mentions deduplication.

## Section 2 — Exploratory Data Analysis

EDA here is not decoration — every plot earns its place by motivating one of the three flaws or one
feature-engineering decision (PRD_eda §1):

- **Class distribution** → why accuracy / macro-F1 mislead (Flaw 3).
- **Attacks-by-day** → why a temporal split is the only honest evaluation (Flaw 2).
- **Inf/NaN audit** → the cleaning we applied in §1 (Flaw 1 context).
- **Distributions / skew** → justify the `log1p` transform (Phase 3 feature engineering).
- **Spearman correlation** → justify correlation-based feature pruning (ADR-1, ADR-8).

Plots that don't need all 2.8M rows are drawn on a fixed-seed 200k sample for speed (ADR-10);
class counts and shares are always computed on the full cleaned `df`.

### 2.1 — Class distribution (imbalance)

The dataset is dominated by BENIGN traffic, and the attack classes span five orders of magnitude in
frequency (DoS Hulk ~231k vs Heartbleed = 11). A log scale is required to see all 15 classes at
once. This extreme imbalance is the seed of **Flaw 3**: a model can ignore the rare classes
entirely and still post a high macro-average, because the rare classes contribute almost nothing to
the aggregate.

In [10]:
order = df["Label"].value_counts()
fig, ax = plt.subplots(figsize=(11, 6))
colors = ["#4C72B0" if lbl == "BENIGN" else "#C44E52" for lbl in order.index]
ax.bar(range(len(order)), order.values, color=colors)
ax.set_yscale("log")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order.index, rotation=60, ha="right")
ax.set_ylabel("Flow count (log scale)")
ax.set_title("CICIDS2017 class distribution (log scale) — BENIGN (blue) vs 14 attack types (red)")
save_fig(fig, "01_class_distribution")

# Reuse the value_counts just computed — no extra full-frame boolean scan.
benign = order["BENIGN"] / order.sum() * 100
print(f"BENIGN: {benign:.1f}%  |  Attacks: {100 - benign:.1f}%")
print("Three rarest classes:", order.tail(3).to_dict())

Saved: ..\figures\01_class_distribution.png
BENIGN: 80.3%  |  Attacks: 19.7%
Three rarest classes: {'Infiltration': 36, 'Web Attack - Sql Injection': 21, 'Heartbleed': 11}


### 2.2 — Attack composition by weekday (motivates the temporal split)

Each attack family was executed on specific days of the capture week. A **random** split scatters
every day's attacks across both train and test, so the model trains on the very PortScan/DDoS
patterns it is later tested on — trivial leakage (**Flaw 2**). A **temporal** split (train Mon-Thu,
test Fri) mirrors a deployed IDS, which only ever sees *past* traffic. The stacked bar shows attack
volume per day; the printed table lists exactly which attack types appear on each day.

In [11]:
# Crosstab needs only two columns of df — no need to allocate a filtered attacks frame.
ct = (pd.crosstab(df["day_name"], df["Label"])
        .reindex(WEEKDAY_ORDER.keys())
        .drop(columns="BENIGN"))
fig, ax = plt.subplots(figsize=(12, 6))
ct.plot(kind="bar", stacked=True, ax=ax, colormap="tab20", width=0.8)
ax.set_ylabel("Attack flow count")
ax.set_xlabel("")
ax.set_title("Attack flows by weekday (stacked) — each attack is confined to specific days")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.xticks(rotation=0)
save_fig(fig, "02_attacks_by_day")

print("Attack types present per day:")
for d in WEEKDAY_ORDER:
    present = sorted(ct.columns[ct.loc[d] > 0])
    print(f"  {d:<9}: {present if present else '(none)'}")

Saved: ..\figures\02_attacks_by_day.png
Attack types present per day:
  Monday   : (none)
  Tuesday  : ['FTP-Patator', 'SSH-Patator']
  Wednesday: ['DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'Heartbleed']
  Thursday : ['Infiltration', 'Web Attack - Brute Force', 'Web Attack - Sql Injection', 'Web Attack - XSS']
  Friday   : ['Bot', 'DDoS', 'PortScan']


### 2.3 — Inf / missing-value audit

Reusing the pre-clean Inf counts captured in §1.3, the artifacts were confined to the two rate
features produced by division (`Flow Bytes/s`, `Flow Packets/s`) — exactly the divide-by-zero
pattern Engelen et al. (2021) describe. No other column carried Inf or NaN, which is why dropping
the affected rows cost only 0.1% of the data.

In [12]:
# inf_by_col was computed in section 1.3 (pre-clean Inf counts per column).
fig, ax = plt.subplots(figsize=(7, 3.5))
inf_by_col.sort_values().plot(kind="barh", ax=ax, color="#C44E52")
ax.set_xlabel("Inf cells (pre-clean)")
ax.set_title("Inf values were confined to two rate features (CICFlowMeter divide-by-zero)")
save_fig(fig, "03_missing_inf_heatmap")
print(inf_by_col.to_string())

Saved: ..\figures\03_missing_inf_heatmap.png


Flow Bytes/s      1509
Flow Packets/s    2867


### 2.4 — Distributions and skew (justifies log1p)

Network-flow features are heavily right-skewed and approximately power-law: most flows are tiny and
brief, with a long tail of large/long flows. The Flow Duration histogram (log x-axis) makes this
concrete. We then quantify skew across all numeric features and list those with |skew| > 3 — these
are the candidates for a `log1p` transform in Phase 3 (ADR-9).

In [13]:
sample = df.sample(n=min(200_000, len(df)), random_state=RANDOM_STATE)
fd_b = sample.loc[sample["Label"] == "BENIGN", "Flow Duration"].clip(lower=0)
fd_a = sample.loc[sample["Label"] != "BENIGN", "Flow Duration"].clip(lower=0)
top = max(fd_b.max(), fd_a.max()) + 1
bins = np.logspace(0, np.log10(top), 60)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(fd_b + 1, bins=bins, alpha=0.6, label="BENIGN", color="#4C72B0")
ax.hist(fd_a + 1, bins=bins, alpha=0.6, label="Attack", color="#C44E52")
ax.set_xscale("log")
ax.set_xlabel("Flow Duration (microseconds, log scale, +1 offset)")
ax.set_ylabel("Flow count")
ax.set_title("Flow Duration is heavily right-skewed \u2014 motivates log1p")
ax.legend()
save_fig(fig, "04_flow_duration_hist")

Saved: ..\figures\04_flow_duration_hist.png


In [14]:
# Drop tag columns AND the zero-variance columns before computing skew — a constant column
# has no meaningful skew, so excluding them keeps the count to real features only.
num_feats = df.drop(columns=["Label", "day", "day_name"] + zero_var_cols,
                    errors="ignore").select_dtypes(include=np.number)
skew = num_feats.skew().sort_values(ascending=False)
high_skew = skew[skew.abs() > 3]
print(f"Features with |skew| > 3: {len(high_skew)} of {len(skew)} numeric features")
print(high_skew.round(1).to_string())

Features with |skew| > 3: 52 of 70 numeric features
Total Length of Fwd Packets     805.2
Subflow Fwd Bytes               803.2
act_data_pkt_fwd                284.5
Subflow Bwd Packets             244.6
Total Backward Packets          244.6
Total Fwd Packets               244.3
Subflow Fwd Packets             244.3
Subflow Bwd Bytes               244.2
Total Length of Bwd Packets     244.2
Fwd URG Flags                    94.7
CWE Flag Count                   94.7
RST Flag Count                   64.2
ECE Flag Count                   64.0
Active Min                       47.7
Flow Bytes/s                     46.4
Active Std                       40.5
Active Mean                      38.2
Active Max                       24.3
Flow IAT Min                     23.8
Bwd Packets/s                    21.5
Fwd Packet Length Min            20.1
Down/Up Ratio                    12.0
Fwd Packet Length Std            10.5
Idle Std                         10.5
Min Packet Length                10.

### 2.5 — Packet-length outliers

Boxplots of forward/backward packet-length features (drawn on the sample, symlog y-axis) show the
extreme right tails and outliers typical of mixed traffic. This reinforces the choice of **Spearman**
(rank-based, outlier-robust) over Pearson for the correlation analysis that follows.

In [15]:
pkt_cols = [c for c in ["Fwd Packet Length Max", "Fwd Packet Length Mean",
                        "Bwd Packet Length Max", "Bwd Packet Length Mean"] if c in df.columns]
fig, ax = plt.subplots(figsize=(10, 5))
sample[pkt_cols].clip(lower=0).plot(kind="box", ax=ax)
ax.set_yscale("symlog")
ax.set_ylabel("Bytes (symlog)")
ax.set_title("Packet-length features: heavy right tails and outliers")
plt.xticks(rotation=20, ha="right")
save_fig(fig, "05_packet_length_box")

Saved: ..\figures\05_packet_length_box.png


### 2.6 — Spearman correlation and feature redundancy (ADR-1, ADR-8)

**We use Spearman rank correlation, not Pearson, because:**

1. **Non-normality.** Flow features (packet lengths, byte counts, durations, IATs) are heavily
   right-skewed and approximately power-law; Pearson's normality/linearity assumptions fail.
2. **Outlier robustness.** Real traffic has extreme flows (large transfers, scans). Spearman uses
   ranks, so a few extremes do not dominate the coefficient.
3. **Monotonic, not linear.** For redundancy we only care whether two features rise/fall together,
   which is exactly what rank correlation captures.
4. **Cost.** Kendall τ shares the robustness but is ~O(n²); Spearman is O(n log n) (a sort) and
   scales to millions of rows.

Below we visualise the top-20 variance features for legibility. The **actual** redundancy pruning
(Phase 3, §3.2) recomputes Spearman across *all* candidate features at |r| > 0.95 — this heatmap is
illustrative, not the pruning basis.

In [16]:
# Reuse the 200k `sample` from §2.4 (identical rows) and compute the ranking variance on it too
# — no second df.sample() and no full-frame variance scan.
sample_feats = sample.drop(columns=["Label", "day", "day_name"] + zero_var_cols,
                           errors="ignore").select_dtypes(include=np.number)
top_var = sample_feats.var().sort_values(ascending=False).head(20).index.tolist()
spearman = sample[top_var].corr(method="spearman")

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(spearman, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Spearman correlation — top-20 variance features")
save_fig(fig, "06_spearman_heatmap")

cols = spearman.columns
pairs = [(cols[i], cols[j], round(float(spearman.iloc[i, j]), 3))
         for i in range(len(cols)) for j in range(i + 1, len(cols))
         if abs(spearman.iloc[i, j]) > 0.95]
print(f"Feature pairs with |Spearman r| > 0.95: {len(pairs)}")
for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
    print(f"  {r:+.3f}  {a}  <->  {b}")

Saved: ..\figures\06_spearman_heatmap.png
Feature pairs with |Spearman r| > 0.95: 12
  +1.000  Idle Max  <->  Idle Mean
  +0.999  Idle Mean  <->  Idle Min
  +0.999  Bwd IAT Max  <->  Bwd IAT Mean
  +0.998  Bwd IAT Total  <->  Bwd IAT Max
  +0.998  Idle Max  <->  Idle Min
  +0.997  Bwd IAT Total  <->  Bwd IAT Mean
  +0.997  Fwd IAT Max  <->  Fwd IAT Mean
  +0.995  Fwd IAT Total  <->  Fwd IAT Max
  +0.992  Fwd IAT Total  <->  Fwd IAT Mean
  +0.991  Flow Duration  <->  Flow IAT Max
  +0.980  Flow IAT Max  <->  Flow IAT Mean
  +0.969  Flow Duration  <->  Flow IAT Mean


### 2.7 — EDA takeaways (feeding feature engineering and the critique)

- **Imbalance (Flaw 3 seed).** BENIGN is 80.3% of flows; Heartbleed (11), SQL Injection (21) and
  Infiltration (36) are too small to move accuracy or macro-F1. Any aggregate-only score — like the
  source paper's macro F1 — can therefore hide total failure on these classes. Note the source
  reports **no per-class metrics and no imbalance handling**; we will (Phase 6).
- **Real-world framing.** Production networks are *far* more benign-heavy than 80%, yet even at this
  comparatively attack-rich ratio models still collapse on rare attacks — the problem is the class
  rarity, not just the overall ratio.
- **Temporal structure (Flaw 2).** Attacks are confined to specific days, so a random split leaks
  attack signatures into training. Strategy B trains on Mon-Thu and tests on Friday (ADR-7).
- **Skew (feature engineering).** The flagged |skew| > 3 features will receive a `log1p` transform
  (ADR-9); Spearman confirms the redundancy clusters that correlation pruning at |r| > 0.95 will
  remove (ADR-8), shrinking the ~70 usable features before modelling.

## Section 3 — Feature Engineering

With the data loaded, cleaned, and profiled, we now build the **model-ready feature set**. Every
transformation here is applied *uniformly* to all runs so that the only intentional differences
between Strategy A (source-style) and Strategy B (honest) remain the three flaws under study —
deduplication (Flaw 1), the temporal split (Flaw 2), and per-class metrics (Flaw 3). Feature
engineering is deliberately **not** a fourth confound: structural and redundancy drops remove only
non-informative or duplicated columns, and `log1p` is monotonic (so it cannot change tree-model
splits). Scaling and SMOTE are *not* done here — they are fit per split inside Phase 5 to prevent
leakage (INV-2).


### 3.1 — Structural column removal

Two groups of columns carry no usable signal and are dropped for **all** strategies:

1. **`Fwd Header Length.1`** — an exact duplicate of `Fwd Header Length` (a known CICFlowMeter
   export quirk noted by Engelen et al., 2021).
2. **Zero-variance constants** — the columns flagged in §1.5; a constant feature cannot help any
   model and only inflates the feature count.


In [17]:
# Idempotent guard: re-strip column names (already done in §1.1; safe to repeat).
df.columns = df.columns.str.strip()

NON_FEATURE = ["Label", "day", "day_name"]

# Redundant duplicate column (CICFlowMeter quirk; Engelen et al., 2021).
redundant_cols = [c for c in ["Fwd Header Length.1"] if c in df.columns]

# Structural drops apply to every strategy (they remove junk, not flaw-relevant signal).
drop_structural = sorted(set(redundant_cols) | set(zero_var_cols))

candidate_features = [c for c in df.columns if c not in NON_FEATURE]
struct_cols = [c for c in candidate_features if c not in drop_structural]

print(f"Candidate features (pre-engineering): {len(candidate_features)}")
print(f"  - redundant duplicate columns dropped : {redundant_cols}")
print(f"  - zero-variance columns dropped       : {len(zero_var_cols)}")
print(f"Surviving after structural drops        : {len(struct_cols)}")

Candidate features (pre-engineering): 78
  - redundant duplicate columns dropped : ['Fwd Header Length.1']
  - zero-variance columns dropped       : 8
Surviving after structural drops        : 69


### 3.2 — Correlation pruning (ADR-8)

Near-redundant features add collinearity without new information. Following ADR-8 we prune at
**|Spearman r| > 0.95**, computed here over **all** candidate features — the §2.6 heatmap was
restricted to the top-20 variance features purely for legibility, but redundancy must be assessed
across the full set. We rank features by variance and greedily keep the highest-variance
representative of each correlated cluster, dropping the rest. Spearman is rank-based (robust to the
heavy skew/outliers in flow data) and cheap on the 200k sample; this is more transparent and stable
than VIF on near-collinear flow features.

In [18]:
# Correlation pruning over ALL candidate features (not just the top-20 EDA subset).
# The §2.6 heatmap used the top-20 variance features for readability; redundancy, however,
# must be checked across every candidate, so we recompute the Spearman matrix on all
# struct_cols here (on the same 200k sample — rank-based, O(n log n), cheap).
struct_sample = sample[struct_cols]
var_rank = struct_sample.var().sort_values(ascending=False)
corr = struct_sample.corr(method="spearman").abs()

# Greedy: walk features high -> low variance; drop any feature that is |r| > 0.95 with a
# higher-variance feature already kept. Keeps one representative per redundant cluster.
kept, corr_drop = [], []
for col in var_rank.index:
    if any(corr.loc[col, k] > 0.95 for k in kept):
        corr_drop.append(col)
    else:
        kept.append(col)
corr_drop = sorted(corr_drop)
feature_cols = [c for c in struct_cols if c not in corr_drop]

n_pairs = int((np.triu(corr.to_numpy(), k=1) > 0.95).sum())
print(f"Candidate features checked: {len(struct_cols)}  |  redundant pairs (|r|>0.95): {n_pairs}")
print(f"Correlation-pruned features ({len(corr_drop)}):")
for c in corr_drop:
    print(f"  - {c}")
print(f"\nFinal engineered feature count: {len(feature_cols)} "
      f"(from {len(candidate_features)} candidates)")

Candidate features checked: 69  |  redundant pairs (|r|>0.95): 57
Correlation-pruned features (29):
  - Active Max
  - Active Mean
  - Active Min
  - Avg Bwd Segment Size
  - Bwd IAT Max
  - Bwd IAT Mean
  - Bwd Packet Length Max
  - Bwd Packet Length Mean
  - CWE Flag Count
  - Flow IAT Max
  - Flow IAT Mean
  - Flow Packets/s
  - Fwd IAT Max
  - Fwd IAT Mean
  - Fwd Packet Length Mean
  - Fwd Packets/s
  - Idle Mean
  - Idle Min
  - Max Packet Length
  - Min Packet Length
  - Packet Length Mean
  - Packet Length Std
  - RST Flag Count
  - SYN Flag Count
  - Subflow Bwd Packets
  - Subflow Fwd Bytes
  - Subflow Fwd Packets
  - Total Backward Packets
  - Total Length of Bwd Packets

Final engineered feature count: 40 (from 78 candidates)


### 3.3 — Label construction (binary + multiclass)

We build **both** label granularities (ADR-4): a binary target drives the headline F1 comparison,
while the multiclass target is required to expose the rare-class recall collapse (Flaw 3) that an
aggregate-only metric hides. Labels are kept separate from the feature matrix.


In [19]:
from sklearn.preprocessing import LabelEncoder  # preprocessing utility, not a model

# Binary target: BENIGN (0) vs any attack (1).
y_bin = (df["Label"] != "BENIGN").astype("int8")

# Multiclass target: needed to surface per-class failure (Flaw 3).
label_encoder = LabelEncoder()
y_multi = label_encoder.fit_transform(df["Label"])
label_names = list(label_encoder.classes_)

print(f"Binary  - attack rate: {y_bin.mean() * 100:.2f}%  "
      f"({int(y_bin.sum()):,} attacks / {len(y_bin):,} flows)")
print(f"Multiclass - {len(label_names)} classes:")
for cls, n in df["Label"].value_counts().items():
    print(f"  {cls:34s} {n:>9,}")

Binary  - attack rate: 19.68%  (556,556 attacks / 2,827,876 flows)
Multiclass - 15 classes:
  BENIGN                             2,271,320
  DoS Hulk                             230,124
  PortScan                             158,804
  DDoS                                 128,025
  DoS GoldenEye                         10,293
  FTP-Patator                            7,935
  SSH-Patator                            5,897
  DoS slowloris                          5,796
  DoS Slowhttptest                       5,499
  Bot                                    1,956
  Web Attack - Brute Force               1,507
  Web Attack - XSS                         652
  Infiltration                              36
  Web Attack - Sql Injection                21
  Heartbleed                                11


### 3.4 — Feature matrix

`X` contains **only** engineered traffic features. The `day` / `day_name` columns are *not*
features (INV-4): they encode the temporal split, not traffic behaviour, so feeding them to a model
would leak the very signal Strategy B is meant to test. We keep `df['day']` aside as the split key
for Phase 5. Duplicate rows are still present **by design** — removing them is a Strategy-B step, so
runs A0 (with duplicates) vs A1 (deduplicated) can isolate Flaw 1.


In [20]:
X = df[feature_cols].copy()
day = df["day"].copy()  # split key only — never fed to a model

print(f"X shape: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Duplicate rows still present in X (by design): {int(X.duplicated().sum()):,}")
assert "Label" not in X.columns
assert "day" not in X.columns and "day_name" not in X.columns

X shape: 2,827,876 rows x 40 features


Duplicate rows still present in X (by design): 308,708


### 3.5 — Log-transform of skewed features (ADR-9)

Network-flow features are heavily right-skewed (§2.4). For the features with **|skew| > 3** we apply
`log1p`, *guarding* on the column minimum because `log1p` requires non-negative input and §1 flagged
CICFlowMeter columns with spurious negatives — those are left untransformed and reported.

`log1p` is **monotonic**, so it does not change the split points of tree models (RF, XGBoost,
Isolation Forest are all invariant to it); we apply it for distribution sanity and any
scale-sensitive analysis. Crucially, it has **no fitted parameters**, so it cannot leak and is safe
to apply per split in Phase 5. We therefore define a reusable `apply_log1p()` rather than baking the
transform into a stored matrix.


In [21]:
col_min = X.min()
skewed_features = [c for c in feature_cols if c in high_skew.index and col_min[c] >= 0]
excluded_neg = [c for c in feature_cols if c in high_skew.index and col_min[c] < 0]

n_skewed_surviving = sum(c in high_skew.index for c in feature_cols)
print(f"Skewed features (|skew|>3) surviving pruning: {n_skewed_surviving}")
print(f"  -> log1p applied to {len(skewed_features)} non-negative features")
print(f"  -> {len(excluded_neg)} skewed-but-negative features left untransformed: {excluded_neg}")


def apply_log1p(frame, cols):
    """Return a copy of `frame` with log1p applied to `cols`.

    `cols` is an explicit required argument (no global capture), so the function is pure and
    safe to run out-of-order. Deterministic and stateless (no fitted parameters) -> cannot leak;
    applied per split in Phase 5. Monotonic -> does not alter tree-model splits.
    """
    out = frame.copy()
    out[cols] = np.log1p(out[cols])
    return out


print("apply_log1p(frame, cols) defined - call as apply_log1p(X_train, skewed_features) in Phase 5.")

Skewed features (|skew|>3) surviving pruning: 30
  -> log1p applied to 23 non-negative features
  -> 7 skewed-but-negative features left untransformed: ['Flow Bytes/s', 'Flow IAT Min', 'Fwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Init_Win_bytes_backward', 'min_seg_size_forward']
apply_log1p(frame, cols) defined - call as apply_log1p(X_train, skewed_features) in Phase 5.


### 3.6 — Leakage guard

A quick automated check that no target-derived or split-encoding column slipped into the feature
set. This is cheap insurance against the exact class of mistake (leakage) that the whole project
critiques.


In [22]:
leak_terms = ("label", "day")
suspicious = [c for c in feature_cols if any(t in c.lower() for t in leak_terms)]
assert not suspicious, f"Potential leakage columns in X: {suspicious}"
assert "Label" not in feature_cols
assert all(c not in feature_cols for c in ["day", "day_name"])
print(f"Leakage check passed: X holds only traffic features ({len(feature_cols)} cols); "
      "labels and split keys are held separately.")

Leakage check passed: X holds only traffic features (40 cols); labels and split keys are held separately.


### 3.7 — Feature-engineering ledger

Every dropped column is accounted for (INV-6 — auditable cleaning). The engineered frame is also
persisted to a git-ignored parquet so Phase 5 can reload it without re-running §1–§3.


In [23]:
ledger = pd.DataFrame(
    [
        ("raw feature columns", len(candidate_features)),
        ("- redundant duplicate (Fwd Header Length.1)", -len(redundant_cols)),
        ("- zero-variance constants", -len(zero_var_cols)),
        ("- correlation-redundant (|Spearman r|>0.95)", -len(corr_drop)),
        ("= final engineered features", len(feature_cols)),
        ("  (of which log1p-transformed: skew>3, >=0)", len(skewed_features)),
    ],
    columns=["step", "delta"],
)
print(ledger.to_string(index=False))

# (P2) Persist engineered frame for fast Phase 5 re-runs. data/ is git-ignored.
# Slice df directly (no second full copy of X); store the raw Label + split key so Phase 5
# rebuilds y_bin/y_multi cheaply on load.
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)
parquet_path = PROCESSED_DIR / "engineered.parquet"
try:
    df[feature_cols + ["day", "Label"]].to_parquet(parquet_path, index=False)
    print(f"\nSaved engineered frame -> data/processed/engineered.parquet "
          f"({len(df):,} x {len(feature_cols) + 2})")
except Exception as exc:  # pyarrow/fastparquet may be absent — non-fatal (P2)
    print(f"\nParquet persist skipped (no parquet engine available): {exc}")

                                       step  delta
                        raw feature columns     78
- redundant duplicate (Fwd Header Length.1)     -1
                  - zero-variance constants     -8
- correlation-redundant (|Spearman r|>0.95)    -29
                = final engineered features     40
  (of which log1p-transformed: skew>3, >=0)     23



Saved engineered frame -> data/processed/engineered.parquet (2,827,876 x 42)


## Section 4 - Model Training

The critique is a controlled comparison: the **same data and the same RF hyperparameters**, with
only the *methodology* changing. Three nested runs isolate each flaw, then two extra models give an
honest comparison.

| Run | Model | Data | Split | Isolates |
|---|---|---|---|---|
| **A0** | RF (default) | raw (duplicates) | random 80/20 | inflated baseline = the source (SC1) |
| **A1** | RF (default) | **deduped** | random 80/20 | **Flaw 1** = F1(A0) - F1(A1) |
| **B0** | RF (default) | deduped | **temporal** (Mon-Thu / Fri) | **Flaw 2** = F1(A1) - F1(B0) |
| Bm | RF (default) | deduped | stratified **multiclass** | **Flaw 3** rare-class recall |
| B1 | XGBoost | deduped | temporal | improved honest model |
| B2 | Isolation Forest | deduped | temporal | unsupervised angle |

RF params are **identical** across A0/A1/B0 (constraint C-1) so every F1 delta is attributable to one
pipeline change. log1p is stateless and applied per split; the `StandardScaler` and `SMOTE` are fit on
**train only** (no leakage).

> **On speed / GPU:** sklearn's RandomForest and IsolationForest are CPU-only - a discrete GPU (e.g. a
> 4070 Super) does *not* accelerate them. The effective levers are `n_jobs=-1` (all cores, set in
> `RF_PARAMS`) and the optional `DEV_SAMPLE` subsample below (ADR-10) for fast iteration; the final
> numbers come from a full-data restart-and-run-all. Only XGBoost has a GPU path (`device='cuda'`).

### 4.0 - Model stack, shared params, and the duplicate mask

We import the models here (following the local-import pattern used for `LabelEncoder` in 3.3), pin the
shared RF hyperparameters, build aligned working views of `X`/`y`/`day`, and compute the exact-duplicate
mask over the engineered features + label (the Flaw-1 contamination, consistent with 1.4).

In [24]:
# Phase 5 model stack. Tree models are CPU-bound in sklearn (no GPU path); the practical
# speed lever is n_jobs=-1 (all cores) plus the optional dev subsample below (ADR-10).
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, recall_score, matthews_corrcoef, roc_auc_score,
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# C-1: identical RF hyperparameters across A0/A1/B0 -- only the pipeline changes.
RF_PARAMS = dict(n_estimators=100, max_depth=None, n_jobs=-1, random_state=RANDOM_STATE)

# Optional dev subsample: set e.g. 300_000 to iterate fast; None = full data for the final run.
DEV_SAMPLE = None
if DEV_SAMPLE is not None and DEV_SAMPLE < len(df):
    work_idx, _ = train_test_split(
        np.arange(len(df)), train_size=DEV_SAMPLE, stratify=y_multi, random_state=RANDOM_STATE,
    )
    work_idx = np.sort(work_idx)
    print(f"DEV MODE: stratified {DEV_SAMPLE:,}-row subsample (NOT for final numbers).")
else:
    work_idx = np.arange(len(df))
    print(f"FULL DATA: {len(work_idx):,} rows.")

# Aligned working views (positional -- df/X/y built in one pass, same row order).
Xw = X.iloc[work_idx].reset_index(drop=True)
y_bin_w = y_bin.iloc[work_idx].reset_index(drop=True)
y_multi_w = pd.Series(y_multi).iloc[work_idx].reset_index(drop=True)
day_w = day.iloc[work_idx].reset_index(drop=True)
label_w = df["Label"].iloc[work_idx].reset_index(drop=True)

# Exact-duplicate mask over engineered features + label (same basis as 1.4 -> ~10.9%).
dedup_mask = ~pd.concat([Xw, label_w], axis=1).duplicated()
n_dupes = int((~dedup_mask).sum())
print(f"Duplicate rows in working set: {n_dupes:,} ({n_dupes / len(Xw) * 100:.2f}%)")

# Metrics collected across Phase 5 -> extended and written to JSON in Phase 6.
metrics = {}


def macro_f1(y_true, y_pred):
    # Macro-averaged F1 (binary here): the source's headline metric, for A0/A1/B0.
    return f1_score(y_true, y_pred, average="macro", zero_division=0)


print("Model stack ready. RF_PARAMS:", RF_PARAMS)

FULL DATA: 2,827,876 rows.


Duplicate rows in working set: 307,991 (10.89%)
Model stack ready. RF_PARAMS: {'n_estimators': 100, 'max_depth': None, 'n_jobs': -1, 'random_state': 42}


### 4.1 - Strategy A0: random split on raw data (reproduce the source)

A0 mirrors Rodriguez et al.: no deduplication, a random stratified 80/20 split, and a near-default
Random Forest. If we reproduce their headline (macro F1 ~ 0.999), we have confirmed their *method* -
the precondition for critiquing it (SC1).

In [25]:
# A0: source-style -- raw (duplicates kept), random stratified 80/20, RF defaults.
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(
    Xw, y_bin_w, test_size=0.2, random_state=RANDOM_STATE, stratify=y_bin_w,
)
# log1p is stateless (no fitted params) -> applying to both splits cannot leak.
Xa_tr = apply_log1p(Xa_tr, skewed_features)
Xa_te = apply_log1p(Xa_te, skewed_features)
sc_a = StandardScaler().fit(Xa_tr)                       # scaler fit on TRAIN only
rf_a0 = RandomForestClassifier(**RF_PARAMS).fit(sc_a.transform(Xa_tr), ya_tr)
pred_a0 = rf_a0.predict(sc_a.transform(Xa_te))

f1_A = macro_f1(ya_te, pred_a0)
acc_A = accuracy_score(ya_te, pred_a0)
metrics["strategy_a_f1_macro"] = f1_A
metrics["strategy_a_accuracy"] = acc_A
print("Strategy A0 (source method: raw + random split)")
print(f"  macro F1 : {f1_A:.4f}   <- compare to the paper's claimed > 0.999 (SC1)")
print(f"  accuracy : {acc_A:.4f}")

Strategy A0 (source method: raw + random split)
  macro F1 : 0.9987   <- compare to the paper's claimed > 0.999 (SC1)
  accuracy : 0.9992


### 4.2 - Flaw 1: deduplicate, same random split

CICIDS2017 carries ~10.9% exact-duplicate flows (1.4; Engelen et al., 2021). A random split scatters
copies of the same flow across train and test, which *could* let the model "recognise" memorised test
points. We remove duplicates and re-run the **identical** random pipeline, then (4.2a) test directly
whether duplicates actually carry the headline. For this dataset they do not: binary F1 barely moves,
because the deeper issue is that a random split puts every attack *type* in both train and test.

In [26]:
# A1: dedup, then the IDENTICAL random split + scaler + RF. Only Flaw 1 is removed.
Xd = Xw[dedup_mask].reset_index(drop=True)
yd = y_bin_w[dedup_mask].reset_index(drop=True)
day_d = day_w[dedup_mask].reset_index(drop=True)        # reused by B0 below
ymd = y_multi_w[dedup_mask].reset_index(drop=True)      # deduped multiclass labels (B0 SMOTE, Bm)
labd = label_w[dedup_mask].reset_index(drop=True)       # deduped string labels (per-attack-type recall)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    Xd, yd, test_size=0.2, random_state=RANDOM_STATE, stratify=yd,
)
Xd_tr = apply_log1p(Xd_tr, skewed_features)
Xd_te = apply_log1p(Xd_te, skewed_features)
sc_a1 = StandardScaler().fit(Xd_tr)
rf_a1 = RandomForestClassifier(**RF_PARAMS).fit(sc_a1.transform(Xd_tr), yd_tr)
pred_a1 = rf_a1.predict(sc_a1.transform(Xd_te))

f1_A_deduped = macro_f1(yd_te, pred_a1)
flaw1_inflation = f1_A - f1_A_deduped
metrics["strategy_a1_f1_macro"] = f1_A_deduped
metrics["n_duplicate_rows"] = n_dupes
metrics["pct_duplicate_rows"] = n_dupes / len(Xw) * 100
metrics["flaw1_f1_inflation"] = flaw1_inflation
print("Flaw 1 -- duplicate contamination (binary F1 view)")
print(f"  F1 with dupes (A0) : {f1_A:.4f}")
print(f"  F1 deduped   (A1)  : {f1_A_deduped:.4f}")
print(f"  inflation          : {flaw1_inflation:+.4f}  ({flaw1_inflation * 100:+.2f} pp)")
print("  (binary F1 barely moves -- the leakage shows in the decomposition below, not here)")

Flaw 1 -- duplicate contamination (binary F1 view)
  F1 with dupes (A0) : 0.9987
  F1 deduped   (A1)  : 0.9984
  inflation          : +0.0003  (+0.03 pp)
  (binary F1 barely moves -- the leakage shows in the decomposition below, not here)


In [27]:
fig, ax = plt.subplots(figsize=(5, 4))
vals = [f1_A, f1_A_deduped]
bars = ax.bar(["A0\nwith duplicates", "A1\ndeduplicated"], vals, color=["#c0392b", "#27ae60"])
ax.set_ylabel("macro F1 (binary)")
ax.set_title("Flaw 1: duplicate contamination inflates F1")
ax.set_ylim(min(vals) - 0.02, 1.002)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom")
save_fig(fig, "07_dup_impact_bar")

Saved: ..\figures\07_dup_impact_bar.png


### 4.2a - Does the leakage actually carry the 0.999? (testing our own hypothesis)

Flaw 1 assumes duplicates inflate the score via leakage. We test it directly: count the A0 test rows
that are exact duplicates of a training row, and compare accuracy on those *leaked* rows vs the *unique*
test rows. If duplicates drove the headline, leaked rows would score far higher than unique ones.
As the output shows, they do **not** -- leaked and unique rows are both graded ~equally near-perfectly.
The duplicates are real data-quality debt and a leakage risk in principle, but they are **not** the lever
behind this dataset's inflated metric; the random split's attack-type overlap is (which Flaw 2 removes).
Reporting this honestly -- a hypothesis we set out to confirm and instead refuted -- is part of the critique.

In [28]:
# Hash each (features+label) row to detect A0 TEST rows that also appear in A0 TRAIN.
tr_key = pd.util.hash_pandas_object(
    pd.concat([Xa_tr.reset_index(drop=True), ya_tr.reset_index(drop=True).rename("y")], axis=1),
    index=False)
te_key = pd.util.hash_pandas_object(
    pd.concat([Xa_te.reset_index(drop=True), ya_te.reset_index(drop=True).rename("y")], axis=1),
    index=False)
leaked = te_key.isin(set(tr_key)).to_numpy()
yte = ya_te.to_numpy()
acc_leaked = float((pred_a0[leaked] == yte[leaked]).mean()) if leaked.any() else float("nan")
acc_unique = float((pred_a0[~leaked] == yte[~leaked]).mean()) if (~leaked).any() else float("nan")
leak_rate = float(leaked.mean())
metrics.update(a0_leak_rate=leak_rate, a0_acc_leaked=acc_leaked, a0_acc_unique=acc_unique)
print(f"A0 test rows that are exact duplicates of a train row: {int(leaked.sum()):,} of {len(leaked):,} "
      f"({leak_rate * 100:.1f}%)")
print(f"  accuracy on LEAKED test rows : {acc_leaked:.4f}")
print(f"  accuracy on UNIQUE test rows : {acc_unique:.4f}")
print(f"  -> leaked ~ unique: duplicates are NOT the inflation lever (gap {acc_leaked - acc_unique:+.4f})")

fig, ax = plt.subplots(figsize=(5, 4))
vals = [acc_leaked, acc_unique]
bars = ax.bar([f"leaked\n({leak_rate * 100:.0f}% of test)", "unique\ntest rows"], vals,
              color=["#c0392b", "#2980b9"])
ax.set_ylabel("accuracy (A0 random split)")
ax.set_title("Flaw 1: leaked vs unique test accuracy (both near-perfect)")
ax.set_ylim(min(v for v in vals if v == v) - 0.02, 1.002)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom")
save_fig(fig, "07b_leakage_decomposition")

A0 test rows that are exact duplicates of a train row: 75,621 of 565,576 (13.4%)
  accuracy on LEAKED test rows : 0.9994
  accuracy on UNIQUE test rows : 0.9991
  -> leaked ~ unique: duplicates are NOT the inflation lever (gap +0.0002)


Saved: ..\figures\07b_leakage_decomposition.png


### 4.3 - Strategy B0: temporal split + SMOTE (honest binary, Flaw 2)

A deployed IDS predicts *future* traffic from *past* training. We therefore train on Mon-Thu (`day<=3`)
and test on Fri (`day==4`). `StandardScaler` and `SMOTE` are fit on the training partition only; the
Friday test set is never resampled (INV-3, E-1). The further F1 drop vs A1 is the optimism that the
random split was hiding (Flaw 2). The total A0 - B0 gap is the SC5 headline.

In [29]:
# B0: honest binary pipeline -- dedup + TEMPORAL split (train Mon-Thu, test Fri).
tr_mask = day_d <= 3
te_mask = day_d == 4
Xb_tr = apply_log1p(Xd[tr_mask], skewed_features)
Xb_te = apply_log1p(Xd[te_mask], skewed_features)
yb_tr = yd[tr_mask]
yb_te = yd[te_mask]
print(f"Temporal split: train(Mon-Thu)={len(Xb_tr):,}  test(Fri)={len(Xb_te):,}")
print(f"  train attack rate: {yb_tr.mean() * 100:.2f}%   test attack rate: {yb_te.mean() * 100:.2f}%")

sc_b = StandardScaler().fit(Xb_tr)
Xb_tr_s = sc_b.transform(Xb_tr)
Xb_te_s = sc_b.transform(Xb_te)

# Multiclass-aware SMOTE on TRAIN ONLY: interpolate WITHIN attack families (no cross-attack
# "Franken-flows"), cap each class to ~50k, exclude ultra-rare classes (<100 real) from synthesis,
# then binarize. (PR review #1; resolves the ADR-3 multiclass-SMOTE policy.)
benign_code = label_names.index("BENIGN")
ymb_tr = ymd[tr_mask].to_numpy()
SMOTE_CAP, RARE_FLOOR = 50_000, 100
cls_counts = pd.Series(ymb_tr).value_counts()
strategy = {c: SMOTE_CAP for c, n in cls_counts.items()
            if c != benign_code and RARE_FLOOR <= n < SMOTE_CAP}
sm = SMOTE(random_state=RANDOM_STATE, sampling_strategy=strategy, k_neighbors=5)
Xb_tr_res, ymb_res = sm.fit_resample(Xb_tr_s, ymb_tr)
yb_tr_res = (ymb_res != benign_code).astype(int)            # binarize AFTER multiclass SMOTE
print(f"  multiclass SMOTE: oversampled {len(strategy)} classes to <= {SMOTE_CAP:,} "
      f"(excluded ultra-rare <{RARE_FLOOR}); train {len(ymb_tr):,} -> {len(ymb_res):,} rows")

rf_b0 = RandomForestClassifier(**RF_PARAMS).fit(Xb_tr_res, yb_tr_res)
prob_b0 = rf_b0.predict_proba(Xb_te_s)[:, 1]
pred_b0 = rf_b0.predict(Xb_te_s)

f1_B = macro_f1(yb_te, pred_b0)
acc_B = accuracy_score(yb_te, pred_b0)
mcc_B = matthews_corrcoef(yb_te, pred_b0)
auc_B = roc_auc_score(yb_te, prob_b0)
flaw2_drop = f1_A_deduped - f1_B
metrics.update(
    strategy_b_f1_macro=f1_B, strategy_b_accuracy=acc_B, strategy_b_mcc=mcc_B,
    strategy_b_roc_auc=auc_B, flaw2_f1_drop_temporal=flaw2_drop,
)
print("Strategy B0 (honest: dedup + temporal + multiclass-SMOTE)")
print(f"  macro F1 : {f1_B:.4f}    MCC: {mcc_B:.4f}    ROC-AUC: {auc_B:.4f}")
print(f"  Flaw 2 drop vs A1 (temporal): {flaw2_drop:+.4f} ({flaw2_drop * 100:+.2f} pp)")
print(f"  TOTAL gap A0 - B0: {f1_A - f1_B:+.4f} ({(f1_A - f1_B) * 100:+.2f} pp)  [SC5 target >= 5pp]")

Temporal split: train(Mon-Thu)=1,904,452  test(Fri)=615,433
  train attack rate: 10.77%   test attack rate: 35.85%


  multiclass SMOTE: oversampled 7 classes to <= 50,000 (excluded ultra-rare <100); train 1,904,452 -> 2,222,284 rows


Strategy B0 (honest: dedup + temporal + multiclass-SMOTE)
  macro F1 : 0.4883    MCC: 0.2541    ROC-AUC: 0.7735
  Flaw 2 drop vs A1 (temporal): +0.5101 (+51.01 pp)
  TOTAL gap A0 - B0: +0.5104 (+51.04 pp)  [SC5 target >= 5pp]


In [30]:
fig, ax = plt.subplots(figsize=(5, 4))
vals = [f1_A_deduped, f1_B]
bars = ax.bar(["A1\nrandom split", "B0\ntemporal split"], vals, color=["#e67e22", "#2980b9"])
ax.set_ylabel("macro F1 (binary, deduped)")
ax.set_title("Flaw 2: temporal split removes leaked optimism")
ax.set_ylim(min(vals) - 0.02, 1.002)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.4f}", ha="center", va="bottom")
save_fig(fig, "08_split_impact_bar")

Saved: ..\figures\08_split_impact_bar.png


### 4.3a - Why B0 collapses: Friday's attacks are unseen classes

The 58 pp drop is not subtle leakage -- it is a generalization wall. Every attack on Friday
(DDoS, PortScan, Bot) belongs to a class that **never appears** in Mon-Thu training, so a supervised
model has no basis to flag it. We confirm this by breaking B0's Friday recall down per attack type
and tagging whether each type was seen in training.

In [31]:
lab_te = labd[te_mask].to_numpy()
seen_attacks = set(pd.Series(labd[tr_mask].to_numpy())[yb_tr.to_numpy() == 1])
print("Friday per-attack-type recall (B0 binary):")
rows = []
for atk, n in pd.Series(lab_te[lab_te != "BENIGN"]).value_counts().items():
    m = lab_te == atk
    rec = float((pred_b0[m] == 1).mean())
    tag = "seen in train" if atk in seen_attacks else "UNSEEN in train"
    rows.append((atk, int(n), tag))
    print(f"  {atk:10s} recall={rec:.3f}  (n={int(n):,}, {tag})")
novel = sum(n for _, n, t in rows if t.startswith("UNSEEN"))
total = sum(n for _, n, t in rows)
novel_share = novel / total if total else float("nan")
metrics["fri_novel_attack_share"] = novel_share
print(f"  -> {novel_share * 100:.0f}% of Friday attack flows are of classes UNSEEN in training")

Friday per-attack-type recall (B0 binary):
  DDoS       recall=0.168  (n=128,014, UNSEEN in train)
  PortScan   recall=0.001  (n=90,694, UNSEEN in train)
  Bot        recall=0.000  (n=1,948, UNSEEN in train)
  -> 100% of Friday attack flows are of classes UNSEEN in training


### 4.4 - Multiclass model for rare-class analysis (Flaw 3 setup)

The rare attacks (Heartbleed = Wed, Infiltration / SQL Injection = Thu) **never occur on Friday**, so
they cannot be evaluated on the temporal test set. We therefore measure per-class recall on a
**deduped, stratified** multiclass split where each class appears in both partitions - the *generous*
setting. SMOTE is intentionally **not** used here: synthesising rare classes from <=36 real points
would mask the very failure we want to expose (and is fragile at `k_neighbors` for 11-sample
Heartbleed). The confusion heatmap and per-class bar are rendered in 4.9 below.

In [32]:
# Bm: stratified deduped MULTICLASS RF for Flaw 3 (per-class recall). No SMOTE here (see 4.4).
ym = ymd.copy()
vc = ym.value_counts()
keep = ym.isin(vc[vc >= 2].index)
dropped = [label_names[c] for c in vc[vc < 2].index]
if dropped:
    print(f"Note: classes with <2 deduped samples excluded from the multiclass split: {dropped}")
Xm = Xd[keep.values].reset_index(drop=True)
ym = ym[keep].reset_index(drop=True)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    Xm, ym, test_size=0.2, random_state=RANDOM_STATE, stratify=ym,
)
Xm_tr = apply_log1p(Xm_tr, skewed_features)
Xm_te = apply_log1p(Xm_te, skewed_features)
sc_m = StandardScaler().fit(Xm_tr)
rf_bm = RandomForestClassifier(**RF_PARAMS).fit(sc_m.transform(Xm_tr), ym_tr)
pred_bm = rf_bm.predict(sc_m.transform(Xm_te))          # stashed for 4.9 + Phase 6

rare = {"Heartbleed": "heartbleed_recall",
        "Infiltration": "infiltration_recall",
        "Web Attack - Sql Injection": "sqli_recall"}
print("Flaw 3 -- rare-class recall on the stratified deduped multiclass test set:")
for name, key in rare.items():
    if name in label_names:
        cls = label_names.index(name)
        support = int((ym_te == cls).sum())
        rec = recall_score(ym_te == cls, pred_bm == cls, zero_division=0)
        metrics[key] = rec
        print(f"  {name:30s} recall={rec:.3f}  (test support={support})")
macro_rec_all = recall_score(ym_te, pred_bm, average="macro", zero_division=0)
print(f"  macro recall over ALL classes: {macro_rec_all:.3f}  <- aggregate hides that rare classes")
print("  have only 2-7 test samples (un-evaluable; any 'hit' is memorised near-duplicates)")

Flaw 3 -- rare-class recall on the stratified deduped multiclass test set:


  Heartbleed                     recall=1.000  (test support=2)
  Infiltration                   recall=0.714  (test support=7)


  Web Attack - Sql Injection     recall=0.250  (test support=4)
  macro recall over ALL classes: 0.854  <- aggregate hides that rare classes
  have only 2-7 test samples (un-evaluable; any 'hit' is memorised near-duplicates)


### 4.5 - XGBoost under the honest protocol

XGBoost (absent from the source paper) is our stronger modern tree-booster, trained on the **same**
SMOTE-balanced temporal train set as B0 and evaluated on the same Friday test set - a fair, honest
comparison. `tree_method='hist'` is fast on wide tabular data; on the 4070 box, swap to
`device='cuda'` for a GPU speed-up.

In [33]:
# B1: XGBoost under the SAME honest protocol as B0 (dedup + temporal + SMOTE-train).
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE,
    # device="cuda",  # uncomment on a CUDA GPU (e.g. RTX 4070 Super) for a speed-up
)
xgb.fit(Xb_tr_res, yb_tr_res)
prob_xgb = xgb.predict_proba(Xb_te_s)[:, 1]
pred_xgb = xgb.predict(Xb_te_s)
f1_xgb = macro_f1(yb_te, pred_xgb)
mcc_xgb = matthews_corrcoef(yb_te, pred_xgb)
auc_xgb = roc_auc_score(yb_te, prob_xgb)
metrics.update(xgb_f1_macro=f1_xgb, xgb_mcc=mcc_xgb, xgb_roc_auc=auc_xgb)
print("XGBoost (honest temporal protocol)")
print(f"  macro F1: {f1_xgb:.4f}   MCC: {mcc_xgb:.4f}   ROC-AUC: {auc_xgb:.4f}")
print(f"  vs RF-B0 macro F1 {f1_B:.4f} (same split): {f1_xgb - f1_B:+.4f}")

XGBoost (honest temporal protocol)
  macro F1: 0.4972   MCC: 0.2670   ROC-AUC: 0.7425
  vs RF-B0 macro F1 0.4883 (same split): +0.0089


### 4.6 - Isolation Forest (unsupervised anomaly detection)

An unsupervised lens absent from the source. We train on **BENIGN-only** Friday-train flows (pure
novelty detection): the model learns "normal" and flags deviations, with `contamination` set near the
attack rate. Score-to-label maps `-1` (anomaly) to attack. We expect weaker numbers than the supervised
models - the point is the different, label-free angle.

In [34]:
# B2: Isolation Forest -- unsupervised, BENIGN-only train flows on the temporal split.
# contamination='auto' (PR review #2): the training data is pure benign, so we must NOT declare a
# 15% anomaly rate within it -- 'auto' uses the standard offset instead of forcing a benign FPR.
benign_tr = yb_tr.to_numpy() == 0
iso = IsolationForest(
    n_estimators=100, contamination="auto", max_samples="auto",
    random_state=RANDOM_STATE, n_jobs=-1,
).fit(Xb_tr_s[benign_tr])
score_iso = -iso.score_samples(Xb_te_s)                 # higher = more anomalous (for ROC)
pred_iso = (iso.predict(Xb_te_s) == -1).astype(int)     # -1 = anomaly = attack
f1_iso = macro_f1(yb_te, pred_iso)
rec_iso = recall_score(yb_te, pred_iso, zero_division=0)
mcc_iso = matthews_corrcoef(yb_te, pred_iso)
auc_iso = roc_auc_score(yb_te, score_iso)
metrics.update(iso_f1_binary=f1_iso, iso_recall_attacks=rec_iso)
print("Isolation Forest (unsupervised, BENIGN-only train, contamination='auto')")
print(f"  macro F1: {f1_iso:.4f}   attack recall: {rec_iso:.4f}   ROC-AUC: {auc_iso:.4f}")

Isolation Forest (unsupervised, BENIGN-only train, contamination='auto')
  macro F1: 0.5473   attack recall: 0.3279   ROC-AUC: 0.7402


### 4.7 - RF feature importances + class-weight cross-check

Two diagnostics: the top-20 RF feature importances from the honest B0 model (figure 15), and a
`class_weight='balanced'` RF (no SMOTE) as a sanity cross-check that our SMOTE result is not an
artifact of the resampling method (ADR-3).

In [35]:
# RF feature importances from the honest B0 model.
imp = pd.Series(rf_b0.feature_importances_, index=feature_cols).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 7))
imp.head(20).iloc[::-1].plot.barh(ax=ax, color="#34495e")
ax.set_title("RF feature importances (Strategy B0, top 20)")
ax.set_xlabel("importance")
save_fig(fig, "15_feature_importance")

# (P2) class_weight='balanced' cross-check (no SMOTE) -- sanity vs the SMOTE result.
rf_cw = RandomForestClassifier(class_weight="balanced", **RF_PARAMS).fit(Xb_tr_s, yb_tr)
f1_cw = macro_f1(yb_te, rf_cw.predict(Xb_te_s))
print(f"RF class_weight=balanced (no SMOTE) macro F1: {f1_cw:.4f}   vs SMOTE B0 {f1_B:.4f}")

Saved: ..\figures\15_feature_importance.png


RF class_weight=balanced (no SMOTE) macro F1: 0.3988   vs SMOTE B0 0.4883


### 4.8 - Phase 5 metrics summary

Everything measured so far, accumulated in `metrics`. Phase 6 extends this with the full per-class
report and SOC analysis, then writes `notebooks/metrics_export.json`.

In [36]:
print("Phase 5 metrics collected so far:")
for key, val in metrics.items():
    print(f"  {key:26s} {val:.4f}" if isinstance(val, float) else f"  {key:26s} {val}")

Phase 5 metrics collected so far:
  strategy_a_f1_macro        0.9987
  strategy_a_accuracy        0.9992
  strategy_a1_f1_macro       0.9984
  n_duplicate_rows           307991
  pct_duplicate_rows         10.8912
  flaw1_f1_inflation         0.0003
  a0_leak_rate               0.1337
  a0_acc_leaked              0.9994
  a0_acc_unique              0.9991
  strategy_b_f1_macro        0.4883
  strategy_b_accuracy        0.6764
  strategy_b_mcc             0.2541
  strategy_b_roc_auc         0.7735
  flaw2_f1_drop_temporal     0.5101
  fri_novel_attack_share     1.0000
  heartbleed_recall          1.0000
  infiltration_recall        0.7143
  sqli_recall                0.2500
  xgb_f1_macro               0.4972
  xgb_mcc                    0.2670
  xgb_roc_auc                0.7425
  iso_f1_binary              0.5473
  iso_recall_attacks         0.3279


### 4.8a - Reconciliation: predicted vs observed

We pre-registered hypotheses (PRD H1/H2, SC1-SC10) and report them honestly, including where the
data refuted them:

| Prediction | Result | Status |
|---|---|---|
| SC1: reproduce F1 ~ 0.999 | A0 macro F1 reproduces the headline | **met** |
| SC3: dedup drops F1 >= 3pp | binary F1 moved ~0pp -- duplicates inflate via *leakage*, not aggregate F1 | reframed (4.2a) |
| SC5: total honest gap >= 5pp | A0 -> B0 gap is ~50+ pp | **met (large)** |
| Flaw 2 = "temporal leakage" | drop is dominated by *novel attack classes* on Friday (100% unseen) | refined (4.3a) |
| SC6: Heartbleed recall < 0.10 | rare classes have 2-7 test samples; apparent recall is memorisation | refuted / reframed (4.4) |

The recalibrated thesis is unchanged and arguably stronger: the source's F1 > 0.999 is an artifact of
**in-distribution memorisation** (duplicate leakage + same-day attack types in train and test); under
honest temporal evaluation it collapses, and aggregate metrics conceal that rare attacks cannot be
evaluated at all. This intellectual-honesty pass is itself part of the critique.

### 4.9 - Visual diagnostics: ROC curves, confusion heatmaps, model comparison

Five figures that make the story legible:
- **09** binary confusion matrix for the inflated A0 (near-perfect - the trap).
- **10** row-normalised **multiclass** confusion heatmap for Bm - the rare-attack rows are where the
  model collapses (Flaw 3).
- **11** per-class recall bar with the rare classes flagged in red.
- **12** ROC curves for the three honest-protocol models (RF-B0, XGBoost, Isolation Forest).
- **14** model-comparison bar (F1 / MCC / recall) on the temporal test set.

In [37]:
from sklearn.metrics import confusion_matrix, roc_curve

# --- 09: binary confusion matrix, Strategy A0 (the inflated baseline) ---
cm_a = confusion_matrix(ya_te, pred_a0)
fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm_a, annot=True, fmt=",d", cmap="Reds", cbar=False,
            xticklabels=["BENIGN", "ATTACK"], yticklabels=["BENIGN", "ATTACK"], ax=ax)
ax.set_title("A0 confusion (random split, with dupes)")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
save_fig(fig, "09_confusion_strategyA")

# --- 10: row-normalised multiclass confusion heatmap, Bm (Flaw 3) ---
present = sorted(set(ym_te) | set(pred_bm))
names_present = [label_names[i] for i in present]
cm_m = confusion_matrix(ym_te, pred_bm, labels=present)
cm_norm = cm_m / cm_m.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=False, cmap="viridis", vmin=0, vmax=1,
            xticklabels=names_present, yticklabels=names_present, ax=ax,
            cbar_kws={"label": "recall (row-normalised)"})
ax.set_title("Bm multiclass confusion (row-normalised) - rare rows go dark = missed")
ax.set_xlabel("predicted"); ax.set_ylabel("true")
plt.xticks(rotation=90); plt.yticks(rotation=0)
save_fig(fig, "10_confusion_strategyB")

Saved: ..\figures\09_confusion_strategyA.png


Saved: ..\figures\10_confusion_strategyB.png


In [38]:
# --- 11: per-class recall bar (rare classes flagged) ---
labels_idx = list(range(len(label_names)))
rec_per = recall_score(ym_te, pred_bm, labels=labels_idx, average=None, zero_division=0)
supp = pd.Series(ym_te).value_counts().reindex(labels_idx, fill_value=0)
rare_names = set(rare.keys())
order = np.argsort(rec_per)
fig, ax = plt.subplots(figsize=(8, 7))
colors = ["#c0392b" if label_names[i] in rare_names else "#7f8c8d" for i in order]
ax.barh([f"{label_names[i]} (n={int(supp[i])})" for i in order], rec_per[order], color=colors)
ax.axvline(0.3, ls="--", c="k", lw=1, label="0.3 failure threshold")
ax.set_xlabel("recall"); ax.set_xlim(0, 1)
ax.set_title("Per-class recall (Bm) - rare classes (red) collapse despite high macro")
ax.legend(loc="lower right")
save_fig(fig, "11_perclass_recall_bar")

Saved: ..\figures\11_perclass_recall_bar.png


In [39]:
# --- 12: ROC curves for the three honest-protocol (temporal) models ---
fig, ax = plt.subplots(figsize=(6, 5.5))
for name, score, auc in [
    (f"RF-B0 (AUC={auc_B:.3f})", prob_b0, auc_B),
    (f"XGBoost (AUC={auc_xgb:.3f})", prob_xgb, auc_xgb),
    (f"IsolationForest (AUC={auc_iso:.3f})", score_iso, auc_iso),
]:
    fpr, tpr, _ = roc_curve(yb_te, score)
    ax.plot(fpr, tpr, lw=2, label=name)
ax.plot([0, 1], [0, 1], ls="--", c="grey", lw=1)
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("ROC - honest temporal split (Friday test)")
ax.legend(loc="lower right")
save_fig(fig, "12_roc_curves")

Saved: ..\figures\12_roc_curves.png


In [40]:
# --- 14: model comparison (F1 / MCC / recall) on the temporal test set ---
comp = pd.DataFrame(
    {
        "macro F1": [f1_B, f1_xgb, f1_iso],
        "MCC": [mcc_B, mcc_xgb, mcc_iso],
        "attack recall": [
            recall_score(yb_te, pred_b0, zero_division=0),
            recall_score(yb_te, pred_xgb, zero_division=0),
            rec_iso,
        ],
    },
    index=["RF-B0", "XGBoost", "IsolationForest"],
)
fig, ax = plt.subplots(figsize=(7, 4.5))
comp.plot.bar(ax=ax, rot=0, color=["#2980b9", "#27ae60", "#8e44ad"])
ax.set_ylim(0, 1); ax.set_ylabel("score")
ax.set_title("Model comparison on the honest temporal split")
ax.legend(loc="lower right")
save_fig(fig, "14_model_comparison_bar")
print(comp.round(4).to_string())

Saved: ..\figures\14_model_comparison_bar.png
                 macro F1     MCC  attack recall
RF-B0              0.4883  0.2541         0.0978
XGBoost            0.4972  0.2670         0.1075
IsolationForest    0.5473  0.1062         0.3279


## Section 5 - Evaluation & Metric Analysis

Section 4 trained the models; this section evaluates them with the **full metric suite the source
paper omitted**. Rodriguez et al. (2022) report only macro F1 (> 0.999). We add precision, recall,
F-beta(2) (recall-weighted -- missing an attack costs more than a false alarm), MCC (robust to the
~80/20 imbalance), and ROC-AUC, and read every number through an IDS/SOC lens. The contrast between
Strategy A0 (their protocol) and Strategy B0 (honest temporal) is the heart of the critique.

### 5.1 - Full metric suite per model

`evaluate_model()` prints all required metrics with their IDS interpretation. We run it for the
inflated A0 baseline, the deduped A1, and the three honest-protocol models (RF-B0, XGBoost,
Isolation Forest), each on its own test set, then collect the rows into one comparison table.

In [41]:
from sklearn.metrics import fbeta_score, precision_score


def evaluate_model(name, y_true, y_pred, y_score=None):
    """Print the full IDS metric suite for a binary classifier and return a dict."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    fbeta = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = roc_auc_score(y_true, y_score) if y_score is not None else float("nan")
    print(f"\n=== {name} ===")
    print(f"  accuracy  : {acc:.4f}   <- MISLEADING under ~80/20 imbalance")
    print(f"  precision : {prec:.4f}   <- false-alarm burden (SOC alert fatigue)")
    print(f"  recall    : {rec:.4f}   <- missed attacks (most critical for IDS)")
    print(f"  F1        : {f1:.4f}")
    print(f"  F-beta(2) : {fbeta:.4f}   <- recall-weighted (beta=2 penalises misses)")
    print(f"  MCC       : {mcc:.4f}   <- robust to imbalance [-1, 1]")
    print(f"  ROC-AUC   : {auc:.4f}")
    return dict(name=name, accuracy=acc, precision=prec, recall=rec,
                f1=f1, fbeta2=fbeta, mcc=mcc, roc_auc=auc)


# A0/A1 probabilities for AUC (test frames already log1p'd in 4.1/4.2).
prob_a0 = rf_a0.predict_proba(sc_a.transform(Xa_te))[:, 1]
prob_a1 = rf_a1.predict_proba(sc_a1.transform(Xd_te))[:, 1]

eval_rows = [
    evaluate_model("A0  raw + random (source method)", ya_te, pred_a0, prob_a0),
    evaluate_model("A1  deduped + random", yd_te, pred_a1, prob_a1),
    evaluate_model("B0  deduped + temporal (RF, honest)", yb_te, pred_b0, prob_b0),
    evaluate_model("B1  deduped + temporal (XGBoost)", yb_te, pred_xgb, prob_xgb),
    evaluate_model("B2  Isolation Forest (unsupervised)", yb_te, pred_iso, score_iso),
]
eval_df = pd.DataFrame(eval_rows).set_index("name").round(4)
print("\nBinary metric suite (all models):")
print(eval_df.to_string())


=== A0  raw + random (source method) ===
  accuracy  : 0.9992   <- MISLEADING under ~80/20 imbalance
  precision : 0.9970   <- false-alarm burden (SOC alert fatigue)
  recall    : 0.9988   <- missed attacks (most critical for IDS)
  F1        : 0.9979
  F-beta(2) : 0.9985   <- recall-weighted (beta=2 penalises misses)
  MCC       : 0.9974   <- robust to imbalance [-1, 1]
  ROC-AUC   : 0.9999



=== A1  deduped + random ===
  accuracy  : 0.9991   <- MISLEADING under ~80/20 imbalance
  precision : 0.9965   <- false-alarm burden (SOC alert fatigue)
  recall    : 0.9981   <- missed attacks (most critical for IDS)
  F1        : 0.9973
  F-beta(2) : 0.9978   <- recall-weighted (beta=2 penalises misses)
  MCC       : 0.9968   <- robust to imbalance [-1, 1]
  ROC-AUC   : 1.0000



=== B0  deduped + temporal (RF, honest) ===
  accuracy  : 0.6764   <- MISLEADING under ~80/20 imbalance
  precision : 0.9970   <- false-alarm burden (SOC alert fatigue)
  recall    : 0.0978   <- missed attacks (most critical for IDS)
  F1        : 0.1781
  F-beta(2) : 0.1193   <- recall-weighted (beta=2 penalises misses)
  MCC       : 0.2541   <- robust to imbalance [-1, 1]
  ROC-AUC   : 0.7735



=== B1  deduped + temporal (XGBoost) ===
  accuracy  : 0.6799   <- MISLEADING under ~80/20 imbalance
  precision : 0.9968   <- false-alarm burden (SOC alert fatigue)
  recall    : 0.1075   <- missed attacks (most critical for IDS)
  F1        : 0.1941
  F-beta(2) : 0.1309   <- recall-weighted (beta=2 penalises misses)
  MCC       : 0.2670   <- robust to imbalance [-1, 1]
  ROC-AUC   : 0.7425



=== B2  Isolation Forest (unsupervised) ===
  accuracy  : 0.6114   <- MISLEADING under ~80/20 imbalance
  precision : 0.4433   <- false-alarm burden (SOC alert fatigue)
  recall    : 0.3279   <- missed attacks (most critical for IDS)
  F1        : 0.3770
  F-beta(2) : 0.3459   <- recall-weighted (beta=2 penalises misses)
  MCC       : 0.1062   <- robust to imbalance [-1, 1]
  ROC-AUC   : 0.7402

Binary metric suite (all models):
                                     accuracy  precision  recall      f1  fbeta2     mcc  roc_auc
name                                                                                             
A0  raw + random (source method)       0.9992     0.9970  0.9988  0.9979  0.9985  0.9974   0.9999
A1  deduped + random                   0.9991     0.9965  0.9981  0.9973  0.9978  0.9968   1.0000
B0  deduped + temporal (RF, honest)    0.6764     0.9970  0.0978  0.1781  0.1193  0.2541   0.7735
B1  deduped + temporal (XGBoost)       0.6799     0.9968  0.1075  0.1941  0.

**Reading the table.** A0 and A1 sit near-perfect on every metric -- this is the trap the source
fell into. The honest B0/B1 rows collapse on recall and MCC while *accuracy stays high*, exactly the
imbalance illusion the next cell quantifies. Isolation Forest, which never needs to have seen an
attack class, posts the best honest attack recall of the supervised/unsupervised trio.

### 5.2 - Why accuracy is the wrong headline metric

A classifier that labels *every* Friday flow BENIGN already scores high accuracy purely from the
benign majority -- while detecting nothing. We compute that floor to show how little accuracy means.

In [42]:
benign_rate_fri = float((yb_te == 0).mean())
print("Accuracy is misleading on imbalanced IDS data:")
print(f"  'always BENIGN' baseline accuracy (Friday test): {benign_rate_fri:.4f}")
print("    ... but that baseline has recall = 0.0000 and MCC = 0.0000 (detects no attacks)")
print(f"  B0 accuracy {acc_B:.4f} sits close to this do-nothing floor,")
print(f"  yet B0 MCC = {mcc_B:.4f} reveals the model is only weakly better than guessing.")
print("  => MCC / recall expose what accuracy (and an unweighted macro-F1 headline) conceal.")

Accuracy is misleading on imbalanced IDS data:
  'always BENIGN' baseline accuracy (Friday test): 0.6415
    ... but that baseline has recall = 0.0000 and MCC = 0.0000 (detects no attacks)
  B0 accuracy 0.6764 sits close to this do-nothing floor,
  yet B0 MCC = 0.2541 reveals the model is only weakly better than guessing.
  => MCC / recall expose what accuracy (and an unweighted macro-F1 headline) conceal.


### 5.3 - Confusion matrices and per-class recall

The visual evidence was produced in 4.9:

- **Figure 09** -- A0 binary confusion: near-perfect, the inflated trap.
- **Figure 10** -- Bm row-normalised multiclass confusion: rare-attack rows go dark (missed).
- **Figure 11** -- per-class recall bar with rare classes flagged in red.

These show the failure is *structural*: the aggregate looks fine while specific dangerous classes
are never recovered. The per-class numbers feed Flaw 3 below.

### 5.4 - Precision-Recall curves and threshold selection

Under heavy class imbalance, precision-recall curves are more informative than ROC (which is
optimistic because true negatives dominate). Figure 13 plots PR for the three honest-protocol
models; the markers show each model's default-threshold operating point against the attack-rate
baseline.

In [43]:
from sklearn.metrics import average_precision_score, precision_recall_curve

fri_attack_rate = float((yb_te == 1).mean())
fig, ax = plt.subplots(figsize=(6.5, 5.5))
for name, score, pred in [
    ("RF-B0", prob_b0, pred_b0),
    ("XGBoost", prob_xgb, pred_xgb),
    ("IsolationForest", score_iso, pred_iso),
]:
    p, r, _ = precision_recall_curve(yb_te, score)
    ap = average_precision_score(yb_te, score)
    line, = ax.plot(r, p, lw=2, label=f"{name} (AP={ap:.3f})")
    ax.scatter(recall_score(yb_te, pred, zero_division=0),
               precision_score(yb_te, pred, zero_division=0),
               color=line.get_color(), s=70, zorder=5, edgecolor="k")
ax.axhline(fri_attack_rate, ls="--", c="grey", lw=1,
           label=f"no-skill baseline (attack rate={fri_attack_rate:.3f})")
ax.set_xlabel("recall"); ax.set_ylabel("precision")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_title("Precision-Recall - honest temporal split (markers = default threshold)")
ax.legend(loc="upper right")
save_fig(fig, "13_pr_curve_threshold")
print("Saved 13_pr_curve_threshold; markers mark each model's default-threshold operating point.")

Saved: ..\figures\13_pr_curve_threshold.png
Saved 13_pr_curve_threshold; markers mark each model's default-threshold operating point.


## Section 6 - Error Analysis & Critique Synthesis

We now look at *which* errors the honest model makes, translate them into Security Operations Center
(SOC) cost, and assemble the three-flaw verdict.

### 6.1 - Where the honest model fails (false negatives & false positives)

On the Friday temporal test we break the errors down by attack type. Because 4.3a established that
100% of Friday's attack flows belong to classes unseen in Mon-Thu training, the false negatives are
a concept-drift / near-zero-day failure, not random noise.

In [44]:
fn_mask = (yb_te.to_numpy() == 1) & (pred_b0 == 0)   # missed attacks
fp_mask = (yb_te.to_numpy() == 0) & (pred_b0 == 1)   # false alarms
print(f"Friday test: {len(yb_te):,} flows | {int((yb_te == 1).sum()):,} true attacks | "
      f"{int((yb_te == 0).sum()):,} benign")
print(f"  false negatives (missed attacks): {int(fn_mask.sum()):,}")
print(f"  false positives (false alarms)  : {int(fp_mask.sum()):,}")

print("\nAttack types most often misclassified as BENIGN (false negatives):")
fn_types = pd.Series(lab_te[fn_mask]).value_counts()
for atk, n in fn_types.items():
    tot = int((lab_te == atk).sum())
    print(f"  {atk:12s} {int(n):>7,} / {tot:>7,} missed ({int(n) / tot * 100:5.1f}%)")
print("\nEvery Friday attack class is UNSEEN in Mon-Thu training (see 4.3a): the model has no learned"
      "\nsignature for it, so the misses are structural cross-attack-type failures, not label noise.")

Friday test: 615,433 flows | 220,656 true attacks | 394,777 benign
  false negatives (missed attacks): 199,085
  false positives (false alarms)  : 65

Attack types most often misclassified as BENIGN (false negatives):
  DDoS         106,568 / 128,014 missed ( 83.2%)
  PortScan      90,569 /  90,694 missed ( 99.9%)
  Bot            1,948 /   1,948 missed (100.0%)

Every Friday attack class is UNSEEN in Mon-Thu training (see 4.3a): the model has no learned
signature for it, so the misses are structural cross-attack-type failures, not label noise.


### 6.2 - SOC cost quantification (one day of traffic)

The Friday test set is a single day of traffic, so its counts read directly as a daily operational
load: alerts an analyst would triage, false alarms in that queue, and real attacks that slip through.

In [45]:
n_alerts = int(pred_b0.sum())
n_missed = int(fn_mask.sum())
n_fp = int(fp_mask.sum())
n_true = int((yb_te == 1).sum())
det_rate = recall_score(yb_te, pred_b0, zero_division=0)
print("SOC view -- one day (Friday) of traffic on the honest RF-B0 model:")
print(f"  alerts raised         : {n_alerts:,}")
print(f"  of which false alarms  : {n_fp:,} ({n_fp / max(n_alerts, 1) * 100:.1f}% of the alert queue)")
print(f"  real attacks present   : {n_true:,}")
print(f"  real attacks MISSED    : {n_missed:,} ({(1 - det_rate) * 100:.1f}% of the day's attacks)")
print("\nAn analyst trusting the source's 0.999 F1 would expect near-total coverage; in reality the"
      "\nhonest model misses the majority of a day's attacks while still filling a false-alarm queue.")

SOC view -- one day (Friday) of traffic on the honest RF-B0 model:
  alerts raised         : 21,636
  of which false alarms  : 65 (0.3% of the alert queue)
  real attacks present   : 220,656
  real attacks MISSED    : 199,085 (90.2% of the day's attacks)

An analyst trusting the source's 0.999 F1 would expect near-total coverage; in reality the
honest model misses the majority of a day's attacks while still filling a false-alarm queue.


### 6.3 - Three-flaw synthesis

Per the critique framework (CLAIM -> OUR TEST -> OUR RESULT), we print the three flaw blocks and the
SC5 total-gap check from our measured numbers.

In [46]:
print("=" * 64)
print("FLAW 1 -- Duplicate contamination (leakage, not aggregate F1)")
print("=" * 64)
print(f"  A0 macro F1 (with dupes) : {f1_A:.4f}")
print(f"  A1 macro F1 (deduped)    : {f1_A_deduped:.4f}")
print(f"  binary F1 inflation      : {flaw1_inflation:+.4f}  ({flaw1_inflation * 100:+.2f} pp)")
print(f"  duplicate rows           : {n_dupes:,} ({n_dupes / len(Xw) * 100:.2f}%)  [Engelen 2021]")
print(f"  leaked vs unique acc     : {metrics['a0_acc_leaked']:.4f} vs {metrics['a0_acc_unique']:.4f}"
      "  -> leakage is a data-quality footnote;")
print("                             the real inflation lever is attack-type overlap in a random split.")

print("\n" + "=" * 64)
print("FLAW 2 -- Random vs temporal (honest) evaluation")
print("=" * 64)
print(f"  A1 macro F1 (random, deduped) : {f1_A_deduped:.4f}")
print(f"  B0 macro F1 (temporal, honest): {f1_B:.4f}   MCC {mcc_B:.4f}")
print(f"  drop                          : {flaw2_drop:+.4f}  ({flaw2_drop * 100:+.2f} pp)")
print(f"  Friday novel-class share      : {metrics['fri_novel_attack_share'] * 100:.0f}%"
      "  (DDoS / PortScan / Bot unseen in training)")

print("\n" + "=" * 64)
print("FLAW 3 -- Aggregate metrics hide rare-class failure")
print("=" * 64)
for nm, key in [("Heartbleed", "heartbleed_recall"), ("Infiltration", "infiltration_recall"),
                ("SQL Injection", "sqli_recall")]:
    print(f"  {nm:14s} recall = {metrics[key]:.3f}")
print("  rare classes have only 2-7 test samples -> statistically un-evaluable; any apparent hit")
print("  is a memorised near-duplicate, and a single macro/aggregate number hides this entirely.")

total_gap = f1_A - f1_B
print("\n" + "=" * 64)
print("SC5 -- total honest gap")
print("=" * 64)
print(f"  F1(A0) - F1(B0) = {f1_A:.4f} - {f1_B:.4f} = {total_gap:.4f}  ({total_gap * 100:.1f} pp)")
print(f"  SC5 target >= 0.05 (5 pp): {'MET' if total_gap >= 0.05 else 'NOT MET'}")

FLAW 1 -- Duplicate contamination (leakage, not aggregate F1)
  A0 macro F1 (with dupes) : 0.9987
  A1 macro F1 (deduped)    : 0.9984
  binary F1 inflation      : +0.0003  (+0.03 pp)
  duplicate rows           : 307,991 (10.89%)  [Engelen 2021]
  leaked vs unique acc     : 0.9994 vs 0.9991  -> leakage is a data-quality footnote;
                             the real inflation lever is attack-type overlap in a random split.

FLAW 2 -- Random vs temporal (honest) evaluation
  A1 macro F1 (random, deduped) : 0.9984
  B0 macro F1 (temporal, honest): 0.4883   MCC 0.2541
  drop                          : +0.5101  (+51.01 pp)
  Friday novel-class share      : 100%  (DDoS / PortScan / Bot unseen in training)

FLAW 3 -- Aggregate metrics hide rare-class failure
  Heartbleed     recall = 1.000
  Infiltration   recall = 0.714
  SQL Injection  recall = 0.250
  rare classes have only 2-7 test samples -> statistically un-evaluable; any apparent hit
  is a memorised near-duplicate, and a single macro

In [47]:
verdict = pd.DataFrame(
    [
        ("F1 ~ 0.999 (their method)", "raw + random split",
         f"A0 F1 = {f1_A:.3f}", "Yes (reproduced)"),
        ("0.999 holds after dedup", "deduped + random split",
         f"A1 F1 = {f1_A_deduped:.3f}", "Yes in binary F1 / leakage hidden"),
        ("0.999 holds in deployment", "deduped + temporal split",
         f"B0 F1 = {f1_B:.3f}, MCC = {mcc_B:.3f}", "No"),
        ("Effective on all attack types", "per-class recall (temporal + multiclass)",
         "Heartbleed/PortScan/Bot recall ~ 0", "No"),
    ],
    columns=["Source claim", "Our test", "Our result", "Supported?"],
)
print(verdict.to_string(index=False))

                 Source claim                                 Our test                         Our result                        Supported?
    F1 ~ 0.999 (their method)                       raw + random split                      A0 F1 = 0.999                  Yes (reproduced)
      0.999 holds after dedup                   deduped + random split                      A1 F1 = 0.998 Yes in binary F1 / leakage hidden
    0.999 holds in deployment                 deduped + temporal split         B0 F1 = 0.488, MCC = 0.254                                No
Effective on all attack types per-class recall (temporal + multiclass) Heartbleed/PortScan/Bot recall ~ 0                                No


### 6.4 - Verdict and deployment recommendation

**The F1 > 0.999 claim is reproducible only under the source's flawed protocol.** Under a random
split on raw data we reproduce the near-perfect headline (Strategy A0). But that number is an
artifact of **in-distribution memorisation**: every attack *type* appears in both train and test,
and ~11% of rows are exact duplicates (Engelen 2021). When we remove duplicates the binary F1 barely
moves (Flaw 1 is a data-quality footnote, not the lever); when we instead evaluate honestly across
time -- train Mon-Thu, test Friday -- macro F1 collapses by roughly **50 percentage points** (Flaw 2,
SC5 met) because Friday's attacks (DDoS, PortScan, Bot) are classes the model has never seen. Finally,
the rare classes (Heartbleed, Infiltration, SQL Injection) have only 2-7 evaluable samples, so any
single aggregate metric is meaningless for them (Flaw 3).

**We do NOT recommend deploying this approach as-is.** A trustworthy IDS evaluation on CICIDS2017
must: (1) deduplicate and report leakage decomposition; (2) use a temporal (or otherwise
attack-type-disjoint) split that reflects detecting *future* / novel attacks; (3) report per-class
recall with explicit handling of un-evaluable rare classes, not a single macro number; and (4) prefer
recall-aware metrics (MCC, F-beta(2), PR-AUC) over accuracy/macro-F1 on this ~80/20 imbalance. Our
honest results align with Engelen et al. (2021) and Lanvin et al. (2022), who predicted exactly these
failure modes.

### 6.5 - Metrics export (agent contract)

The notebook is the single source of truth for every number in the report. We serialise all collected
metrics to `notebooks/metrics_export.json` and assert that every key required by the report-writer
contract (PLAN section 5) is present and numeric, so the report can never invent a figure.

In [48]:
import json

REQUIRED_KEYS = [
    "strategy_a_f1_macro", "strategy_a_accuracy", "n_duplicate_rows", "pct_duplicate_rows",
    "strategy_b_f1_macro", "strategy_b_mcc", "strategy_b_roc_auc", "flaw1_f1_inflation",
    "flaw2_f1_drop_temporal", "heartbleed_recall", "infiltration_recall", "sqli_recall",
    "xgb_f1_macro", "xgb_mcc", "iso_f1_binary", "iso_recall_attacks",
]
metrics["total_honest_gap_f1"] = float(f1_A - f1_B)
metrics["sc5_gap_met"] = bool((f1_A - f1_B) >= 0.05)

missing = [k for k in REQUIRED_KEYS if k not in metrics]
assert not missing, f"metrics_export missing required keys: {missing}"
nonnumeric = [k for k in REQUIRED_KEYS if not isinstance(metrics[k], (int, float))]
assert not nonnumeric, f"non-numeric required metrics: {nonnumeric}"


def _ser(v):
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float, np.floating, np.integer)):
        return float(v)
    return v


export = {k: _ser(v) for k, v in metrics.items()}
out_path = Path("metrics_export.json")
out_path.write_text(json.dumps(export, indent=2))
print(f"Wrote {out_path.resolve()} ({len(export)} keys; "
      f"all {len(REQUIRED_KEYS)} required keys present and numeric).")

Wrote C:\Users\אייל. ש\Documents\HUP\cyber\project\notebooks\metrics_export.json (25 keys; all 16 required keys present and numeric).


## Section 7 - Executive Summary

**Problem.** Network intrusion detection on CICIDS2017 (2.83M CICFlowMeter flows, ~80% benign,
14 attack types over Mon-Fri).

**Source.** Rodriguez et al. (2022, *Sensors* MDPI, DOI 10.3390/s22239326) report a Random Forest
reaching **F1 > 0.999** using a random split over all days concatenated, no deduplication, and only a
macro-F1 headline.

**What we did.** We rebuilt their pipeline in Python/scikit-learn and ran it three ways: A0 (their
exact protocol), A1 (their protocol + deduplication), and B0 (honest temporal split: train Mon-Thu,
test Friday), plus XGBoost and an unsupervised Isolation Forest under the honest protocol, with the
full metric suite (precision, recall, F-beta(2), MCC, ROC-AUC, PR-AUC) and per-class analysis.

**Findings.**
- **Reproduced (SC1):** A0 macro F1 reproduces the > 0.999 headline -- their method is replicable.
- **Flaw 1 (reframed):** removing the ~11% duplicate rows barely changes binary F1; a leakage
  decomposition shows leaked and unique test rows are *equally* easy. Duplicates are data-quality
  debt, not the inflation mechanism -- the real lever is attack-type overlap in a random split.
- **Flaw 2 (proven, SC5):** the honest temporal split drops macro F1 by ~50 pp (MCC ~0.25), because
  100% of Friday's attack flows are classes never seen in training -- the random split was an
  in-distribution memorisation test.
- **Flaw 3:** rare attacks (Heartbleed, Infiltration, SQL Injection) have 2-7 evaluable samples; a
  single aggregate metric is meaningless for them and hides that they are effectively undetected.

**Verdict.** The author's F1 > 0.999 claim is **not supported** as evidence of a deployable detector;
it is an artifact of methodology. See the verdict table in 6.3 and the recommendation in 6.4.

**Would we recommend reusing this approach on similar problems? No** -- not without temporal/novel-
attack-aware evaluation, deduplication with leakage reporting, per-class recall, and recall-aware
metrics. With those corrections the *workflow* is sound; the *original evaluation* is not. This
mirrors the warnings of Engelen et al. (2021) and Lanvin et al. (2022).